In [1]:
import pandas as pd
import glob
import numpy as np

In [8]:
def one_hot_with_na(df, columns):
    df_encoded = df.copy()
    for col in columns:
        dummies = pd.get_dummies(df[col], prefix=f"ohe_{col}")
        # Set rows corresponding to NaN in original column to NaN in dummies
        dummies[df[col].isna()] = np.nan
        df_encoded = pd.concat([df_encoded.drop(columns=[col]), dummies], axis=1)
    return df_encoded

In [11]:
target_dir = "simulated_ohe_data"

In [ ]:
# Iterate over all simulated dataframes to be converted.
for file in glob.glob(f"*.tsv"):
    # Open tabular data file.
    df = pd.read_csv(file, sep='\t', index_col=0)
    # Convert to actual NANs.
    df.replace({-99999.0 : np.nan}, inplace=True)
    # Subset to non-binary data.
    cat_df = df[df['type']=='cat'].copy()
    cat_df.drop(columns=['type'], inplace=True)
    cat_df = cat_df.T.copy()

    non_bin_cols = [col for col in cat_df.columns if cat_df[col].nunique(dropna=True) > 2]
    non_bin_df = cat_df[non_bin_cols].copy()
    
    # OHE non-binary columns.
    all_non_bin_cols = [col for col in non_bin_df.columns]
    ohe_df = one_hot_with_na(non_bin_df, all_non_bin_cols)
    
    # Merge all parts of original df again.
    df.drop(columns=['type'], inplace=True)
    df = df.T.copy()
    missing_cols = list(set(df.columns) - set(all_non_bin_cols))
    final_df = pd.concat([ohe_df, df[missing_cols]], axis=1)
    
    final_df.to_csv(f"../{target_dir}/{file}", sep='\t', index=True)

    